[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/09_data_aggregation/09_5_Filter.ipynb)

# 09.5: `filter()`: Keeping Groups by Condition

You already know how to filter individual rows with boolean indexing: `df[df["lifeExp"] > 70]`. That keeps every row where the condition is true for that row.

But some questions are about groups, not individual rows. "Keep only continents where the median GDP exceeds 10,000" is not a row-level question; it is a question about the aggregate behavior of an entire group. Boolean indexing cannot express it cleanly because the condition is computed across all rows in a group, not per row.

Groupby `filter()` solves this: it applies a condition to each group as a whole, and keeps all rows from the groups that satisfy it.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="notebook")

df = pd.read_parquet("gapminder.parquet")
df2007 = df[df["year"] == 2007].copy().reset_index(drop=True)
df2007.head()

## Why boolean indexing is the wrong tool for group conditions

Suppose you want to study only the continents where the median 2007 GDP per capita exceeds $10,000. Let's see what boolean indexing gives you compared to what you actually want.

In [2]:
# First, check the actual median GDP per continent
df2007.groupby("continent")["gdpPercap"].median().sort_values(ascending=False).round(0)

continent
Oceania     29810.0
Europe      28054.0
Americas     8948.0
Asia         4471.0
Africa       1452.0
Name: gdpPercap, dtype: float64

Europe and Oceania have median GDP above $20,000. The Americas are just above $7,000. Africa and Asia are below $2,000. So the condition "median GDP > 10,000" should keep Europe and Oceania, and drop Africa, Asia, and the Americas.

Now watch what a row-level boolean filter does:

In [3]:
# Row-level filter: keeps individual rows where gdpPercap > 10000
row_filtered = df2007[df2007["gdpPercap"] > 10000]
print("Continents remaining after row filter:")
print(row_filtered["continent"].value_counts())

Continents remaining after row filter:
continent
Europe      25
Asia        13
Americas     9
Africa       5
Oceania      2
Name: count, dtype: int64


The row-level filter kept individual rich countries from Africa and Asia, but it did not answer the group question. We wanted to keep entire continents where the median is high, but instead we got a patchwork of individual wealthy countries scattered across all continents. That is a different question and a different answer.

## `groupby().filter()`: the right tool for group conditions

`filter()` takes a function that receives the sub-DataFrame for each group and must return a single boolean: `True` to keep the entire group, `False` to drop every row in it.

In [4]:
# Keep only continents where the median GDP per capita exceeds 10,000
high_gdp = df2007.groupby("continent").filter(
    lambda g: g["gdpPercap"].median() > 10000
)

print("Continents remaining after group filter:")
print(high_gdp["continent"].value_counts())

Continents remaining after group filter:
continent
Europe     30
Oceania     2
Name: count, dtype: int64


Only Europe and Oceania remain: all rows for each of those continents, and nothing from any other continent. The `lambda g:` receives the full sub-DataFrame for each continent. Calling `g["gdpPercap"].median()` computes the median across all countries in that continent. If the result exceeds 10,000, every row in that continent is kept. If not, every row is dropped.

Compare to the row-level filter: there, individual African or Asian countries with high GDP were kept. Here, no rows from those continents remain at all, because the group as a whole did not meet the condition.

## Filtering on group size

A common use of `filter()` is to remove small groups before computing statistics that are unreliable with few observations. Oceania has only 2 countries; computing a standard deviation across 2 data points is not meaningful.

`len(g)` inside the lambda gives the number of rows in the group.

In [5]:
# Keep only continents with at least 10 countries
large_continents = df2007.groupby("continent").filter(lambda g: len(g) >= 10)

print("Country counts per continent after size filter:")
print(large_continents["continent"].value_counts())

Country counts per continent after size filter:
continent
Africa      52
Asia        33
Europe      30
Americas    25
Name: count, dtype: int64


Oceania's two countries are gone. Africa, Asia, Americas, and Europe remain. When computing continent-level standard deviations or percentile ranges, the results for these four continents are more reliable than a statistic computed on two data points.

## Filtering on temporal conditions

`filter()` works on the full Gapminder dataset too, not just the 2007 snapshot. This lets you ask questions like "which continents had every country above a threshold in every year since 1982?"

In [6]:
# Keep only countries where every observation from 1977 onward has lifeExp > 60
df_recent = df[df["year"] >= 1977].copy()

consistent = df_recent.groupby("country").filter(
    lambda g: g["lifeExp"].min() > 60
)

print(f"Countries with lifeExp > 60 in every year from 1977 onward: {consistent['country'].nunique()}")
print(consistent.groupby("continent")["country"].nunique().sort_values(ascending=False))

Countries with lifeExp > 60 in every year from 1977 onward: 69
continent
Europe      29
Asia        18
Americas    18
Africa       2
Oceania      2
Name: country, dtype: int64


The `g["lifeExp"].min() > 60` condition asks whether the lowest life expectancy that country ever recorded from 1977 onward was still above 60. 69 countries pass this test. Europe has 29, Asia and the Americas each have 18, and only 2 African countries qualify, consistent with how broadly the AIDS epidemic and other factors lowered life expectancy across the continent in the 1990s and 2000s.

## Chaining `filter()` with `agg()`

`filter()` returns a plain DataFrame containing only the rows that passed the group condition. That DataFrame can immediately be piped into a `groupby().agg()` for a focused summary.

In [7]:
# Among continents with at least 10 countries:
# compute mean and std of life expectancy in 2007
(
    df2007
    .groupby("continent")
    .filter(lambda g: len(g) >= 10)
    .groupby("continent")["lifeExp"]
    .agg(mean="mean", std="std")
    .round(1)
    .sort_values("mean", ascending=False)
)

,mean,std
continent,,
Europe,77.6,3.0
Americas,73.6,4.4
Asia,70.7,8.0
Africa,54.8,9.6


The chain reads left to right: filter out small continents, then summarize the remaining ones. Filtering then aggregating is the most common use of `filter()` in practice.

## What's next

You have now used `agg()` to collapse groups into summaries, `transform()` to add group statistics back to every row, and `filter()` to keep or remove entire groups by condition. Notebook 09.6 introduces `pivot_table()`, a different interface that produces wide-format aggregations directly, often more convenient than the groupby-then-unstack pattern from notebook 09.2.